In [26]:
import os
import random
import argparse
import sys
sys.path.append(os.path.abspath('..'))
from Tools.generalCV import *

import torch
import os
import cv2
import numpy as np
from PIL import Image
import torchvision.transforms.functional as F
from torchvision.transforms.functional import InterpolationMode

set the random seed

In [27]:
def set_seed(seed_value=42):
    """Set seed for reproducibility."""
    random.seed(seed_value)  # Python random module
    np.random.seed(seed_value)  # Numpy library
    torch.manual_seed(seed_value)  # Torch

    # if using CUDA
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value)  # if using multi-GPU
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed()


In [28]:
example_image_folder = "/home/chiara_piemontese/UltraBones100k/7_test/my_spine_frames_7"

OUTPUT_MASK_DIR = "/home/chiara_piemontese/UltraBones100k/7_test/my_spine_masks7_repo"
OUTPUT_OVERLAY_DIR = "/home/chiara_piemontese/UltraBones100k/7_test/my_spine_overlays7_repo"

import os
os.makedirs(OUTPUT_MASK_DIR, exist_ok=True)
os.makedirs(OUTPUT_OVERLAY_DIR, exist_ok=True)


In [29]:
import matplotlib.pyplot as plt


In [30]:
def main_pure_image(example_image_folder):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # load the model train on specimen: [1,3,4,5,6,9,10,11,12,13,14]
    model_path = "models/train_on_1_3_4_5_6_9_10_11_12_13_14/epoch_100.pth"
    model = torch.load(model_path, map_location=device) 
    model = model.to(device)
    # hotfix per mismatch di versione SMP: disattiva il controllo della shape
    if hasattr(model, "check_input_shape"):
        model.check_input_shape = lambda x: None
    # hotfix 2: aggiungi l'attributo mancante nei blocchi FPN
    for m in model.modules():
        if m.__class__.__name__ == "FPNBlock" and not hasattr(m, "interpolation_mode"):
            m.interpolation_mode = "nearest"   # o "bilinear", ma "nearest" è lo standard qui

    for file in sorted(os.listdir(example_image_folder)):   # meglio sorted
        img_file = os.path.join(example_image_folder, file)
    
        img_cv2 = cv2.imread(img_file, cv2.IMREAD_GRAYSCALE)
        H, W = img_cv2.shape[:2]  # dimensioni originali
    
        img = Image.open(img_file).convert('L')
        x = F.to_tensor(img)
        x = F.resize(x, [256, 256], interpolation=InterpolationMode.BILINEAR)
        x = F.normalize(x, mean=0.17475835978984833, std=0.16475939750671387).unsqueeze(0)
    
        with torch.no_grad():
            y = torch.sigmoid(model(x.to(device)))
    
        # maschera binaria 0/1 (meglio di 0/255 come label map)
        pred_256 = (y[0, 0] > 0.5).cpu().numpy().astype(np.uint8)  # (256,256) valori 0/1
    
        # riportala a (H,W) con nearest
        pred_hw = cv2.resize(pred_256, (W, H), interpolation=cv2.INTER_NEAREST)  # (H,W) 0/1
    
        base = os.path.splitext(file)[0]
        cv2.imwrite(os.path.join(OUTPUT_MASK_DIR, f"{base}.png"), (pred_hw * 255).astype(np.uint8))
    
        # overlay coerente con originale
        ov_img = overlap_image_with_label(img_cv2, (pred_hw * 255).astype(np.uint8))
        cv2.imwrite(os.path.join(OUTPUT_OVERLAY_DIR, f"{base}_overlay.png"), ov_img)


In [31]:
main_pure_image(example_image_folder)

Using device: cuda


/tmp/ipykernel_3207563/3904545414.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model = torch.load(model_path, map_location=device)


In [33]:
import cv2, numpy as np
m = cv2.imread("/home/chiara_piemontese/UltraBones100k/7_test/my_spine_masks7_repo/Ultrasound 1003.png", cv2.IMREAD_UNCHANGED)
print(m.dtype, m.shape, m.min(), m.max(), np.unique(m)[:20])

uint8 (512, 128) 0 255 [  0 255]
